In [10]:
import pandas as pd
import numpy as np
import wrds
import statsmodels.api as sm

from pathlib import Path

In [11]:
OUT_DIR = Path('../data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [12]:
final = pd.read_parquet('../Returns/rets_vars_ff_dec.parquet')

In [13]:
DIR_PROC    = Path("../data/processed");       DIR_PROC.mkdir(parents=True, exist_ok=True)

read_path = DIR_PROC / "df_lm_avg_baseline_wti_gind.parquet"

emissions = pd.read_parquet(
    read_path,
    engine="pyarrow"
)

emissions.rename(columns={'year':'fyear'}, inplace=True)

emissions['sc1'] = emissions['sc1_disclosed']
emissions['sc1'] = np.where(emissions.sc1.isna(), emissions.sc1_estimated, emissions.sc1)
emissions['sc1_disclosed_filled'] = emissions['sc1_disclosed_complete']

In [15]:
emissions_exchg = emissions.loc[emissions["exchg"].isin([11, 12, 14])].reset_index(drop=True).copy()

In [16]:
emissions_exchg.reset_index(drop=True).to_parquet(
    OUT_DIR / "df_lm_avg_baseline_wti_gind_exchg.parquet",
    engine="pyarrow")

In [17]:
final_emissions = final.dropna().reset_index(drop=True).merge(emissions[['gvkey', 'exchg','fyear', 'sc1', 'sc1_disclosed', 'sc1_estimated', 'sc1_ours', 'sc1_adj','sc1_disclosed_filled','gsector']], on=['gvkey', 'fyear'], how='left')
final_emissions = final_emissions.dropna(subset=['sc1', 'sc1_disclosed', 'sc1_estimated', 'sc1_ours','sc1_adj','sc1_disclosed_filled'], how='all').reset_index(drop=True)

final_emissions_exchg = final.dropna().reset_index(drop=True).merge(emissions_exchg[['gvkey', 'exchg','fyear', 'sc1', 'sc1_disclosed', 'sc1_estimated', 'sc1_ours', 'sc1_adj','sc1_disclosed_filled','gsector']], on=['gvkey', 'fyear'], how='left')
final_emissions_exchg = final_emissions_exchg.dropna(subset=['sc1', 'sc1_disclosed', 'sc1_estimated', 'sc1_ours','sc1_adj','sc1_disclosed_filled'], how='all').reset_index(drop=True)

In [8]:
final_emissions_exchg.to_csv(
    OUT_DIR / "df_lm_baseline_wti_gind_rets_ff_dec_exchg.csv",
    index=False
)

In [9]:
final_emissions.to_csv(
    OUT_DIR / "df_lm_baseline_wti_gind_rets_ff_dec.csv",
    index=False
)

### Only once

In [ ]:
db = wrds.Connection(wrds_username='gcrippa4') # Replace with your WRDS username

Loading library list...
Done


In [ ]:
# ---------- Config ----------
GVKEY_FILE = "../Returns/gvkeys.txt"       # 
DATE_CUTOFF = "2010-01-01"      # 

# Timing choice for B/M & CRSP-year alignment:
#   'ff_dec'  -> use CRSP December ME of calendar year t (Fama–French convention)
#   'fye_mon' -> use CRSP ME in the firm's fiscal year-end month
TIMING = 'ff_dec'   # change to 'fye_mon' if you prefer fiscal YE month

In [ ]:
# ---------- Read GVKEY list & pad to 6 ----------
gvkeys = (
    pd.read_csv(GVKEY_FILE, header=None)[0]
    .astype(str).str.strip().str.zfill(6)
    .tolist()
)
if not gvkeys:
    raise ValueError("No GVKEYs found in gvkeys.txt.")
gvlist_sql = ",".join("'" + g + "'" for g in gvkeys) 

In [ ]:
query_funda = f"""
select gvkey, datadate, fyear, datafmt, indfmt, popsrc, consol,
       sale, at, ni, ceq, capx, ppent, dltt, dlc,
       epspx, txditc, txdb, itcb, seq, pstk, pstkrv, pstkl
from comp.funda
where indfmt='INDL' and datafmt='STD' and popsrc='D' and consol='C'
  and datadate >= '{DATE_CUTOFF}'
  and gvkey in ({gvlist_sql})
"""
f = db.raw_sql(query_funda, date_cols=['datadate']).sort_values(['gvkey','fyear']).reset_index(drop=True)

In [ ]:
# ----- Add all variables 

#  ---------- Helpers ----------
def sdiv(num, den):
    den = den.astype(float)
    out = num.astype(float) / den
    out[(den <= 0) | np.isclose(den, 0.0) | den.isna()] = np.nan
    return out

def slog(x):
    x = x.astype(float)
    return pd.Series(np.where(x > 0, np.log(x), np.nan), index=x.index)

def lag_by(df, by, col, n=1):
    return df.groupby(by, group_keys=False)[col].shift(n)
# ---------- Clean denoms & lags ----------
for col in ['sale','at','ceq','ppent']:
    if col in f.columns:
        f.loc[f[col] <= 0, col] = np.nan

f['L_sale']  = lag_by(f, ['gvkey'], 'sale', 1)
f['L_epspx'] = lag_by(f, ['gvkey'], 'epspx', 1)

# ---------- Build accounting metrics ----------
f['leverage']      = sdiv(f['dltt']+f['dlc'],  f['at'])             # (DLTT + DLC) / AT
f['salesgr']       = sdiv(f['sale']  - f['L_sale'],  f['L_sale'])
f['epsgr']         = sdiv(f['epspx'] - f['L_epspx'], f['L_epspx'])
f['log_ppe']       = slog(f['ppent'])
f['invest_a']      = sdiv(f['capx'],  f['at'])             # CAPX / AT

# Book Equity (BE) for B/M (to be paired with CRSP ME)
txditc0  = f['txditc'].fillna(0.0)
f['pstk_adj'] = f[['pstkrv','pstkl','pstk']].bfill(axis=1).iloc[:,0]     # PSTKRV -> PSTKL -> PSTK
f['deferred'] = np.where(
    f['txditc'].notna(), 
    f['txditc'],
    (f[['txdb', 'itcb']].sum(axis=1))
)
f['deferred'] = f['deferred'].fillna(0)

f['seq_f'] = np.where(
    (f['seq'].notna()) & (f['seq'] > 0),
    f['seq'],
    np.where(
        f['ceq'].notna(),
        f['ceq'] + f['pstk'].fillna(0),
        np.nan
    )
)

f['be'] = f['seq_f'] + f['deferred'] - f['pstk_adj']

f['roe'] = sdiv(f['ni'], f['be'])  

# Keep a fundamental frame
acct_cols = [
    'gvkey','datadate','fyear',
    'roe',
    'invest_a','leverage','log_ppe',
    'salesgr','epsgr','be'
]
acct = f[acct_cols].copy()

In [ ]:
# Now, let us attach permno to link with CRSP
ccm = db.raw_sql(f"""
    select gvkey, lpermno as permno, linktype, linkprim, linkdt, linkenddt
    from crsp.ccmxpf_linktable
    where gvkey in ({gvlist_sql})
""", date_cols=['linkdt','linkenddt'])
ccm = ccm[ccm['linktype'].isin(['LU','LC','LN','LS','LX','LD'])]
ccm = ccm[ccm['linkprim'].isin(['P','C'])]
ccm['linkenddt'] = ccm['linkenddt'].fillna(pd.Timestamp('2099-12-31'))

In [ ]:
f_ccm = acct.merge(ccm, on = ['gvkey'])
f_ccm = f_ccm[(f_ccm['datadate'] >= f_ccm['linkdt']) & (f_ccm['datadate'] <= f_ccm['linkenddt'])]
f_ccm = f_ccm.dropna(subset=['permno']).drop_duplicates(['gvkey','fyear','permno'])
f_ccm.reset_index(drop=True, inplace=True)

In [ ]:
# ---- Download CRSP data  --- 
permno_sql = ",".join("'" + p + "'" for p in list(f_ccm.permno.astype('int').astype('str').unique()))

msf = db.raw_sql(f"""
    select permno, date as mdate, ret, retx, prc, shrout
    from crsp.msf
    where date >= '2009-01-01'
        and permno in ({permno_sql})
""", date_cols=['mdate'])

msi = db.raw_sql("""
    select date as mdate, vwretd as mktret
    from crsp.msi
    where date >= '2009-01-01'
""", date_cols=['mdate'])

In [ ]:
crsp = msf.merge(msi, on='mdate', how='left')
for col in ['ret', 'retx', 'prc', 'shrout', 'mktret']:
    crsp[col] = crsp[col].astype(float)
crsp['me'] = crsp['prc'].abs() * (crsp['shrout'] * 1000.0)       # ME in dollars
crsp['yyyymm'] = crsp['mdate'].dt.to_period('M')
crsp['yyyy'] = crsp['mdate'].dt.to_period('Y')
crsp = crsp.sort_values(['permno', 'mdate']).copy()

In [ ]:
crsp['vol_12m'] = (
    crsp.groupby('permno')['ret']
        .rolling(12, min_periods=12)
        .std()
        .reset_index(level=0, drop=True)
)

In [ ]:
crsp['mom_12m_excl1'] = (
    (1 + crsp['ret'].shift(1))  # skip last month
    .groupby(crsp['permno'])
    .rolling(11, min_periods=11)
    .apply(np.prod, raw=True)
    .reset_index(level=0, drop=True)
    - 1.0
)

In [ ]:
def beta_ols(ri, rm):
    x = rm.values
    y = ri.values
    ok = ~np.isnan(x) & ~np.isnan(y)
    if ok.sum() < 6:
        return np.nan
    X = sm.add_constant(x[ok])
    b = sm.OLS(y[ok], X).fit()
    return b.params[1]

In [ ]:
agg_rows = []
for (g, yr), grp in crsp.groupby(['permno','yyyy'], sort=False):
    grp = grp.sort_values('mdate')
    ri = grp['ret'].tail(12)
    rm = grp['mktret'].tail(12)

    beta12  = beta_ols(ri, rm)

    agg_rows.append((g, yr, beta12))

In [ ]:
betas = pd.DataFrame(agg_rows, columns=['permno','yyyy','beta12'])
crsp = crsp.merge(betas)

In [ ]:
# ------- Merge Everything -------- 
crsp['fyear'] = crsp.yyyy.astype('str').astype('int')
final = crsp.merge(f_ccm)
final['gvkey'] = final['gvkey'].astype(int)
final['bm'] = sdiv(final['be'], final['me'])*100000
final['roe'] = final['roe']*100
final['ret'] = final['ret']*100

In [ ]:
# Ensure date is datetime and extract year
hhi = pd.read_csv('../Returns/HHI.csv')

hhi['datadate'] = pd.to_datetime(hhi['datadate'])
hhi['year'] = hhi['datadate'].dt.year

# Optional but recommended: drop segments with missing or zero sales
hhi = hhi[hhi['sales'] > 0]

# Step 1: Total firm sales in each year
hhi['firm_total_sales'] = hhi.groupby(['gvkey', 'year'])['sales'].transform('sum')

# Step 2: Segment share within the firm
hhi['segment_share'] = hhi['sales'] / hhi['firm_total_sales']

# Step 3: Compute within-firm HHI (sum of squared shares)
# This gives HHI on 0–1 scale. Multiply by 10,000 to match standard reporting (0–10,000)
firm_hhi = (
    hhi.groupby(['gvkey', 'year'])['segment_share']
    .apply(lambda x: (x**2).sum()) 
    .reset_index(name='hhi')
)

final['year'] = pd.to_datetime(final['datadate']).dt.year

final = final.merge(firm_hhi, on=['gvkey', 'year'], how='left')

In [ ]:
# final.to_parquet('../Returns/rets_vars_fye_mon.parquet', index=False)
final.to_parquet('../Returns/rets_vars_ff_dec.parquet', index=False)

In [ ]:
emissions = pd.read_parquet(
    "../data/processed/df_lm_avg_baseline.parquet",
    engine="pyarrow"
)
emissions.rename(columns={'year':'fyear'}, inplace=True)

emissions_gam = pd.read_parquet(
    "../data/processed/df_gam_avg_baseline.parquet",
    engine="pyarrow"
)
emissions_gam.rename(columns={'year':'fyear'}, inplace=True)

In [ ]:
emissions['sc1'] = emissions['sc1_disclosed']
emissions['sc1'] = np.where(emissions.sc1.isna(), emissions.sc1_estimated, emissions.sc1)
emissions['sc1_disclosed_filled'] = emissions['sc1_disclosed_complete']

emissions_gam['sc1'] = emissions_gam['sc1_disclosed']
emissions_gam['sc1'] = np.where(emissions_gam.sc1.isna(), emissions_gam.sc1_estimated, emissions_gam.sc1)
emissions_gam['sc1_disclosed_filled'] = emissions_gam['sc1_disclosed_complete']

In [ ]:
final_emissions = final.dropna().reset_index(drop=True).merge(emissions[['gvkey', 'fyear', 'sc1', 'sc1_disclosed', 'sc1_estimated', 'sc1_ours','gsector']], on=['gvkey', 'fyear'], how='left')
final_emissions_gam = final.dropna().reset_index(drop=True).merge(emissions_gam[['gvkey', 'fyear', 'sc1', 'sc1_disclosed', 'sc1_estimated', 'sc1_ours','gsector']], on=['gvkey', 'fyear'], how='left')

In [ ]:
final_emissions = final_emissions.dropna(subset=['sc1', 'sc1_disclosed', 'sc1_estimated', 'sc1_ours'], how='all').reset_index(drop=True)
final_emissions_gam = final_emissions_gam.dropna(subset=['sc1', 'sc1_disclosed', 'sc1_estimated', 'sc1_ours'], how='all').reset_index(drop=True)

In [ ]:
final_emissions.to_csv("../data/processed/df_lm_avg_rets.csv")
final_emissions_gam.to_csv("../data/processed/df_gam_avg_rets.csv")